In [ ]:
import json
import shutil
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from pathlib import Path

In [ ]:
sections_dir = Path("../data/json/security")
sections = []

for json_file in sorted(sections_dir.glob("*.json")):
    file_sections = json.loads(json_file.read_text(encoding="utf-8"))
    sections.extend(file_sections)
    print(f"Loaded {len(file_sections):>3} sections  ←  {json_file.name}")

print(f"\nTotal sections loaded: {len(sections)}")

In [ ]:
from itertools import groupby

CHUNK_SIZE            = 3
OVERLAP               = 1
STRIDE                = CHUNK_SIZE - OVERLAP   # = 2
MAX_CHARS_PER_SECTION = 800

# Boilerplate headings that appear identically in every Spring guide — skip them
SKIP_HEADINGS = {
    "What you'll build",
    "How to complete this guide",
    "What you'll need",
    "Summary"
}

chunks = []

# Sort ensures groupby works even if sections from different files are interleaved
sorted_sections = sorted(sections, key=lambda s: s["source"])

for source, group in groupby(sorted_sections, key=lambda s: s["source"]):
    doc_sections = [
        s for s in group
        if s["content"].strip() and s.get("heading", "") not in SKIP_HEADINGS
    ]
    for i in range(0, len(doc_sections), STRIDE):
        window = doc_sections[i : i + CHUNK_SIZE]
        content = "\n\n".join(
            (f"## {s['heading']}\n" if s["heading"] else "") +
            s["content"].strip()[:MAX_CHARS_PER_SECTION]
            for s in window
        )
        if not content.strip():
            continue
        chunks.append(Document(
            page_content=content,
            metadata={
                "headings": " | ".join(s["heading"] for s in window if s["heading"]),
                "title":    window[0]["title"],
                "source":   source,
            }
        ))

print(f"Created {len(chunks)} chunks  ({CHUNK_SIZE} sections/chunk, {OVERLAP} overlap)")

In [ ]:
import ollama

response = ollama.embed(
    model='nomic-embed-text',
    input=chunks[0].page_content,
)
print(f"{response.embeddings} : {chunks[0].metadata['source']}")

In [ ]:
# Clear old Chroma DB before re-embedding to avoid duplicates
shutil.rmtree("./chroma_db/security", ignore_errors=True)
print("Cleared old vector store.")

In [ ]:
# Embed chunks and store in Chroma (persisted to disk)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectors_dir = "./chroma_db/security"

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=vectors_dir,
    collection_name="spring_security"
)

print(f"Stored {vector_store._collection.count()} embeddings in '{vectors_dir}'")

In [ ]:
# Sanity check: run a test similarity search
query = "Authentication with Spring Security"
results = vector_store.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source : {doc.metadata.get('source', 'unknown')}")
    print(f"Page   : {doc.metadata.get('page', '?')}")
    print(f"Content: {doc.page_content[:100]}")
    print()